# PSSD Treatment Outcomes — r/PSSD

## Abstract

We analyzed 902 treatment reports from 220 users on r/PSSD to answer: **what treatments help or harm people with Post-SSRI Sexual Dysfunction?** SSRIs and SNRIs — which *caused* PSSD — dominate the dataset (62% of mentions) with overwhelmingly negative sentiment, but are excluded from recovery recommendations since they reflect causal context, not recovery outcomes. Among recovery-oriented treatments, antihistamines (67% positive, n=6), ketogenic diet (83% positive, n=5), microdosing (78% positive, n=5), tadalafil (71% positive, n=5), and cabergoline (42% positive, n=9) show the most promising signals. Sample sizes are small (max 18 users per non-SSRI drug), so all findings are preliminary. These results reflect self-reported community data from March–April 2026, not clinical trials.

## 1. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sqlite3
import os
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats as sp_stats
from scipy.stats import binomtest
from IPython.display import display, HTML

# Database path
DB_PATH = os.path.join(os.path.dirname(os.path.abspath("__file__")), "..", "pssd.db")
if not os.path.exists(DB_PATH):
    DB_PATH = "pssd.db"
conn = sqlite3.connect(DB_PATH)

# Sentiment encoding
SENTIMENT_SCORE = {"positive": 1.0, "mixed": 0.5, "neutral": 0.0, "negative": -1.0}

# Plotting style
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 7)
plt.rcParams["figure.dpi"] = 100

COLORS = {"positive": "#2ecc71", "mixed": "#d5d8dc", "neutral": "#d5d8dc", "negative": "#e74c3c"}

# ── SSRI / SNRI / antidepressant names: these CAUSED PSSD ──
# They are causal-context contamination and must be filtered from recovery analyses.
CAUSAL_DRUGS = {
    "ssri", "sertraline", "lexapro", "fluoxetine", "escitalopram",
    "prozac", "paroxetine", "citalopram", "snri", "duloxetine",
    "vortioxetine", "antidepressant", "zoloft", "paxil", "celexa",
    "fluvoxamine", "venlafaxine", "desvenlafaxine", "effexor", "cymbalta",
    "brintellix"
}

# Generic / non-actionable terms to filter from charts
GENERIC_NAMES = {
    "supplements", "medication", "treatment", "therapy", "supplement",
    "drug", "drugs", "medicine", "medications", "antidepressants",
    "ssris", "snris", "75mg", "seed", "psychiatric medications",
    "dopaminergic drugs", "d2 agonist", "ointment"
}

## 2. Data Exploration

Before analyzing treatment outcomes, we examine the shape and quality of the dataset: how many users, reports, and treatments are available, and what the overall sentiment landscape looks like.

In [ ]:
# ── Dataset overview ──
table_counts = {}
for table in ["users", "posts", "treatment", "treatment_reports"]:
    table_counts[table] = pd.read_sql(f"SELECT COUNT(*) as n FROM {table}", conn).iloc[0, 0]

# Date range
date_range = pd.read_sql(
    "SELECT MIN(post_date) as earliest, MAX(post_date) as latest FROM posts WHERE post_date IS NOT NULL", conn
).iloc[0]
d_start = datetime.datetime.fromtimestamp(date_range["earliest"])
d_end = datetime.datetime.fromtimestamp(date_range["latest"])
n_months = round((d_end - d_start).days / 30.44, 1)

n_users_with_reports = pd.read_sql(
    "SELECT COUNT(DISTINCT user_id) as n FROM treatment_reports", conn
).iloc[0, 0]

# Sentiment distribution
sentiment_dist = pd.read_sql("""
    SELECT sentiment, COUNT(*) as reports
    FROM treatment_reports
    GROUP BY sentiment
    ORDER BY reports DESC
""", conn)

display(HTML(f"""
<div style='background:#f8f9fa; padding:20px; border-radius:8px; margin-bottom:15px;'>
    <h3 style='margin-top:0; color:#2c3e50;'>Dataset Overview — r/PSSD</h3>
    <table style='font-size:15px; border-collapse:collapse;'>
        <tr><td style='padding:4px 20px 4px 0; font-weight:bold;'>Users</td><td>{table_counts['users']:,}</td></tr>
        <tr><td style='padding:4px 20px 4px 0; font-weight:bold;'>Posts & comments</td><td>{table_counts['posts']:,}</td></tr>
        <tr><td style='padding:4px 20px 4px 0; font-weight:bold;'>Users with treatment reports</td><td>{n_users_with_reports}</td></tr>
        <tr><td style='padding:4px 20px 4px 0; font-weight:bold;'>Treatment reports</td><td>{table_counts['treatment_reports']:,}</td></tr>
        <tr><td style='padding:4px 20px 4px 0; font-weight:bold;'>Unique treatments</td><td>{table_counts['treatment']:,}</td></tr>
        <tr><td style='padding:4px 20px 4px 0; font-weight:bold;'>Data covers</td><td>{d_start.strftime('%Y-%m-%d')} to {d_end.strftime('%Y-%m-%d')} ({n_months} months)</td></tr>
    </table>
</div>
"""))

### Overall sentiment distribution

The PSSD community skews heavily negative: 62% of all treatment reports carry negative sentiment. This is expected — the subreddit exists because SSRIs harmed these people, and SSRI/antidepressant mentions dominate the dataset.

In [ ]:
# ── Sentiment distribution pie chart ──
fig, ax = plt.subplots(figsize=(7, 5))
color_map = {"negative": "#e74c3c", "positive": "#2ecc71", "mixed": "#f39c12", "neutral": "#95a5a6"}
pie_colors = [color_map.get(s, "#bdc3c7") for s in sentiment_dist["sentiment"]]

wedges, texts, autotexts = ax.pie(
    sentiment_dist["reports"], labels=sentiment_dist["sentiment"],
    colors=pie_colors, autopct="%1.0f%%", startangle=140,
    pctdistance=0.75, textprops={"fontsize": 12}
)
for t in autotexts:
    t.set_fontweight("bold")
    t.set_color("white")
ax.set_title("Overall Treatment Sentiment Distribution (902 reports)", fontsize=14, fontweight="bold", pad=15)
plt.tight_layout()
plt.show()

62% of reports are negative, 26% positive, and 11% mixed. This imbalance is driven by SSRI/SNRI mentions — the drugs that *caused* PSSD — which account for the bulk of negative sentiment. The next sections separate causal-context drugs from recovery-oriented treatments.

## 3. What Caused PSSD: The Causal-Context Drugs

SSRIs, SNRIs, and the generic term "antidepressant" appear frequently in r/PSSD because they **caused** the condition. Their negative sentiment reflects the harm that brought users to this community, not a failed recovery attempt. We show them here for completeness but **exclude them from all recovery analyses and recommendations**.

In [ ]:
# ── Load all treatment reports with drug names ──
df_all = pd.read_sql("""
    SELECT
        tr.user_id,
        t.canonical_name AS drug,
        tr.sentiment AS sentiment_label,
        tr.signal_strength
    FROM treatment_reports tr
    JOIN treatment t ON t.id = tr.drug_id
""", conn)
df_all["sentiment_score"] = df_all["sentiment_label"].map(SENTIMENT_SCORE)
df_all["drug_lower"] = df_all["drug"].str.lower()

# Tag causal vs recovery
df_all["is_causal"] = df_all["drug_lower"].isin(CAUSAL_DRUGS)
df_all["is_generic"] = df_all["drug_lower"].isin(GENERIC_NAMES)

# ── Causal drug table ──
df_causal = df_all[df_all["is_causal"]].copy()

causal_user = df_causal.groupby(["drug", "user_id"]).agg(
    avg_sentiment=("sentiment_score", "mean"),
    n_reports=("sentiment_score", "count")
).reset_index()

causal_summary = causal_user.groupby("drug").agg(
    users=("user_id", "nunique"),
    mean_sentiment=("avg_sentiment", "mean"),
    pct_positive=("avg_sentiment", lambda x: (x > 0.7).mean() * 100),
    pct_negative=("avg_sentiment", lambda x: (x < -0.7).mean() * 100)
).reset_index().sort_values("users", ascending=False)

causal_summary.columns = ["Drug", "Users", "Mean Sentiment", "% Positive", "% Negative"]
causal_summary["Mean Sentiment"] = causal_summary["Mean Sentiment"].round(2)
causal_summary["% Positive"] = causal_summary["% Positive"].round(1)
causal_summary["% Negative"] = causal_summary["% Negative"].round(1)

display(HTML("<h4>Causal-Context Drugs (SSRIs/SNRIs that caused PSSD)</h4>"))
display(HTML("<p><em>These are shown for context only. They are excluded from recovery recommendations because negative sentiment reflects the harm that caused PSSD, not a failed recovery attempt.</em></p>"))
display(causal_summary.head(15).style.hide(axis='index').set_properties(**{"text-align": "center"}).set_table_styles([{"selector": "th", "props": [("text-align", "center")]}]))

In [ ]:
# ── Diverging bar chart for causal drugs ──
causal_plot = causal_user.copy()
causal_plot["outcome"] = causal_plot["avg_sentiment"].apply(
    lambda x: "positive" if x > 0.7 else ("negative" if x < -0.7 else "mixed/neutral")
)

causal_outcomes = causal_plot.groupby(["drug", "outcome"]).size().unstack(fill_value=0)
for col in ["positive", "mixed/neutral", "negative"]:
    if col not in causal_outcomes.columns:
        causal_outcomes[col] = 0
causal_outcomes["total"] = causal_outcomes.sum(axis=1)
causal_outcomes = causal_outcomes[causal_outcomes["total"] >= 3].sort_values("total", ascending=True)

pct_pos = (causal_outcomes["positive"] / causal_outcomes["total"] * 100).values
pct_neg = (causal_outcomes["negative"] / causal_outcomes["total"] * 100).values
pct_mix = (causal_outcomes["mixed/neutral"] / causal_outcomes["total"] * 100).values
drugs_c = causal_outcomes.index.values
y_c = np.arange(len(drugs_c))
counts_c = causal_outcomes["total"].values

fig, ax = plt.subplots(figsize=(11, 5))
ax.barh(y_c, -pct_mix, height=0.6, color="#d5d8dc", label="Mixed/Neutral", edgecolor="white", linewidth=0.5)
ax.barh(y_c, -pct_neg, height=0.6, left=-pct_mix, color="#e74c3c", label="Negative", edgecolor="white", linewidth=0.5)
ax.barh(y_c, pct_pos, height=0.6, color="#2ecc71", label="Positive", edgecolor="white", linewidth=0.5)
ax.axvline(x=0, color="black", linewidth=0.8)

for i, (d, n) in enumerate(zip(drugs_c, counts_c)):
    ax.text(max(pct_pos[i], 5) + 2, i, f"n={n}", va="center", fontsize=9, color="#555")

ax.set_yticks(y_c)
ax.set_yticklabels(drugs_c, fontsize=10)
ax.set_xlabel("% of users", fontsize=11)
ax.set_title("Causal-Context Drugs: What Caused PSSD", fontsize=13, fontweight="bold", pad=12)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{abs(x):.0f}%"))
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=3, frameon=False, fontsize=10)
fig.subplots_adjust(bottom=0.18)
plt.tight_layout()
plt.show()

Every SSRI/SNRI shows overwhelmingly negative sentiment: sertraline (96% negative), paroxetine (100%), vortioxetine (90%), and the generic term "SSRI" (85%). The small positive fraction (6–12%) likely reflects users noting partial recovery after discontinuation, not endorsement of the drug.

**What this means:** These drugs are the *reason* people are on r/PSSD. Their negative sentiment is not a treatment outcome — it is the disease context. They are excluded from all recovery analyses below.

## 4. Recovery Treatments: What Helps or Harms

With causal-context drugs and generic category names filtered out, we examine what people with PSSD are actually trying for recovery, and whether they report positive or negative outcomes.

In [ ]:
# ── Filter to recovery treatments (non-causal, non-generic, 3+ users) ──
df_recovery = df_all[(~df_all["is_causal"]) & (~df_all["is_generic"])].copy()

# User-level aggregation (one data point per user per drug)
recovery_user = df_recovery.groupby(["drug", "user_id"]).agg(
    avg_sentiment=("sentiment_score", "mean"),
    n_reports=("sentiment_score", "count")
).reset_index()

recovery_user["outcome"] = recovery_user["avg_sentiment"].apply(
    lambda x: "positive" if x > 0.7 else ("negative" if x < -0.7 else "mixed/neutral")
)

# Drug-level summary
recovery_summary = recovery_user.groupby("drug").agg(
    users=("user_id", "nunique"),
    mean_sentiment=("avg_sentiment", "mean"),
    pct_positive=("outcome", lambda x: (x == "positive").mean() * 100),
    pct_negative=("outcome", lambda x: (x == "negative").mean() * 100),
    pct_mixed=("outcome", lambda x: (x == "mixed/neutral").mean() * 100)
).reset_index().sort_values("users", ascending=False)

# Only drugs with 3+ users for statistical relevance
recovery_3plus = recovery_summary[recovery_summary["users"] >= 3].copy()

display_df = recovery_3plus[["drug", "users", "mean_sentiment", "pct_positive", "pct_negative", "pct_mixed"]].copy()
display_df.columns = ["Treatment", "Users", "Mean Sentiment", "% Positive", "% Negative", "% Mixed"]
display_df["Mean Sentiment"] = display_df["Mean Sentiment"].round(2)
display_df["% Positive"] = display_df["% Positive"].round(1)
display_df["% Negative"] = display_df["% Negative"].round(1)
display_df["% Mixed"] = display_df["% Mixed"].round(1)

display(HTML("<h4>Recovery Treatments (3+ users, causal-context drugs excluded)</h4>"))
display(display_df.head(25).style.hide(axis='index')
    .background_gradient(subset=["Mean Sentiment"], cmap="RdYlGn", vmin=-1, vmax=1)
    .set_properties(**{"text-align": "center"})
    .set_table_styles([{"selector": "th", "props": [("text-align", "center")]}])
)

In [ ]:
# ── Diverging bar chart: recovery treatments ──
plot_recovery = recovery_3plus.sort_values("mean_sentiment", ascending=True).copy()

drugs_r = plot_recovery["drug"].values
y_r = np.arange(len(drugs_r))
pct_pos_r = plot_recovery["pct_positive"].values
pct_neg_r = plot_recovery["pct_negative"].values
pct_mix_r = plot_recovery["pct_mixed"].values
users_r = plot_recovery["users"].values

fig, ax = plt.subplots(figsize=(12, max(8, len(drugs_r) * 0.38)))

# Mixed innermost (adjacent to center), Negative outermost
ax.barh(y_r, -pct_mix_r, height=0.65, color="#d5d8dc", label="Mixed/Neutral", edgecolor="white", linewidth=0.5)
ax.barh(y_r, -pct_neg_r, height=0.65, left=-pct_mix_r, color="#e74c3c", label="Negative", edgecolor="white", linewidth=0.5)
ax.barh(y_r, pct_pos_r, height=0.65, color="#2ecc71", label="Positive", edgecolor="white", linewidth=0.5)
ax.axvline(x=0, color="black", linewidth=0.8)

for i in range(len(drugs_r)):
    ax.text(max(pct_pos_r[i], 3) + 2, i, f"n={users_r[i]}", va="center", fontsize=8, color="#555")

ax.set_yticks(y_r)
ax.set_yticklabels(drugs_r, fontsize=9)
ax.set_xlabel("% of users", fontsize=11)
ax.set_title("Recovery Treatment Outcomes — r/PSSD (3+ users, SSRIs excluded)",
             fontsize=13, fontweight="bold", pad=12)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{abs(x):.0f}%"))
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.06), ncol=3, frameon=False, fontsize=10)
fig.subplots_adjust(bottom=0.10)
plt.tight_layout()
plt.show()

The diverging bar chart reveals a clear spectrum. Treatments cluster into three groups:

**Net positive:** Ketogenic diet (83%), microdosing (78%), tadalafil (71%), antihistamines (67%), amphetamine (75%), pramipexole (50%), cabergoline (42%), and bupropion (38%) show more positive than negative outcomes.

**Net negative:** Antipsychotics (87% negative), atomoxetine (100%), olanzapine (100%), dextromethorphan/DXM (78–80%), probiotics (70%), gabapentin (60%), and HCG (80%) are associated with mostly negative reports.

**Balanced/unclear:** Buspirone (33%/33%), cyproheptadine (30%/40%), and weed (37%/50%) show roughly even positive/negative splits.

**What this means for patients:** Dopamine-acting drugs (cabergoline, bupropion, pramipexole, amphetamine), PDE5 inhibitors (tadalafil), dietary changes (ketogenic diet), and psychedelics (microdosing) cluster toward the positive end. Drugs that further suppress serotonergic or central nervous system activity tend to show negative outcomes.

## 5. Statistical Testing: Binomial Tests

For each recovery treatment with 3+ users, we test whether the positive outcome rate significantly differs from 50% (chance). With small samples, the exact binomial test is the most appropriate choice.

In [ ]:
# ── Binomial tests on recovery treatments ──
binom_results = []
for _, row in recovery_3plus.iterrows():
    drug = row["drug"]
    drug_users = recovery_user[recovery_user["drug"] == drug]
    n = len(drug_users)
    n_pos = (drug_users["outcome"] == "positive").sum()
    n_neg = (drug_users["outcome"] == "negative").sum()
    test = binomtest(n_pos, n, p=0.5, alternative="two-sided")
    ci = test.proportion_ci(confidence_level=0.95, method="wilson")
    binom_results.append({
        "Treatment": drug,
        "Users": n,
        "Positive": n_pos,
        "Negative": n_neg,
        "% Positive": round(n_pos / n * 100, 1),
        "95% CI": f"{ci.low*100:.0f}–{ci.high*100:.0f}%",
        "p-value": round(test.pvalue, 4),
        "Sig.": "*" if test.pvalue < 0.05 else ("†" if test.pvalue < 0.10 else "")
    })

binom_df = pd.DataFrame(binom_results).sort_values("% Positive", ascending=False)

display(HTML("<h4>Binomial Test: Positive Rate vs. 50% Baseline</h4>"))
display(HTML("<p><em>* = p &lt; 0.05 &nbsp;&nbsp; † = p &lt; 0.10 &nbsp;&nbsp; Wilson 95% CI for proportion</em></p>"))
display(binom_df.head(30).style.hide(axis='index')
    .set_properties(**{"text-align": "center"})
    .set_table_styles([{"selector": "th", "props": [("text-align", "center")]}])
)

With maximum sample sizes of 18 users per drug, statistical power is limited. Most treatments do not reach significance at p < 0.05. This is expected with community-sourced data in a niche condition.

**What this means:** Lack of statistical significance does not mean a treatment doesn't work — it means we don't have enough data to be confident. The pattern of results (direction and consistency) still carries useful information, especially when multiple drugs in the same class point the same way.

## 6. Forest Plot: Mean Sentiment with Confidence Intervals

A forest plot shows the mean sentiment for each treatment along with 95% confidence intervals. Narrower intervals indicate more consistent reports; wider intervals mean more uncertainty. Treatments are sorted by mean sentiment.

In [ ]:
# ── Forest plot for recovery treatments ──
forest_data = []
for _, row in recovery_3plus.iterrows():
    drug = row["drug"]
    scores = recovery_user[recovery_user["drug"] == drug]["avg_sentiment"].values
    n = len(scores)
    mean = scores.mean()
    if n >= 2:
        se = sp_stats.sem(scores)
        ci = se * sp_stats.t.ppf(0.975, n - 1)
    else:
        ci = 0
    forest_data.append({"drug": drug, "mean": mean, "ci": ci, "n": n})

forest_df = pd.DataFrame(forest_data).sort_values("mean", ascending=True)

fig, ax = plt.subplots(figsize=(10, max(8, len(forest_df) * 0.38)))
y_f = np.arange(len(forest_df))
means = forest_df["mean"].values
cis = forest_df["ci"].values
ns = forest_df["n"].values
drug_names = forest_df["drug"].values

# Color by direction
dot_colors = ["#2ecc71" if m > 0.25 else "#e74c3c" if m < -0.25 else "#95a5a6" for m in means]

ax.errorbar(means, y_f, xerr=cis, fmt="none", ecolor="#777", elinewidth=1.2, capsize=3, zorder=1)
ax.scatter(means, y_f, c=dot_colors, s=65, zorder=2, edgecolors="white", linewidth=0.5)
ax.axvline(x=0, color="gray", linestyle="--", alpha=0.5, linewidth=1)

for i in range(len(forest_df)):
    ax.text(1.15, i, f"n={ns[i]}", va="center", fontsize=8, color="#777")

ax.set_yticks(y_f)
ax.set_yticklabels(drug_names, fontsize=9)
ax.set_xlabel("Mean Sentiment Score (95% CI)", fontsize=11)
ax.set_title("Recovery Treatment Precision — Forest Plot", fontsize=13, fontweight="bold", pad=12)
ax.set_xlim(-1.4, 1.5)
plt.tight_layout()
plt.show()

The forest plot reveals important nuances beyond the bar chart:

- **Ketogenic diet, microdosing, and tadalafil** have confidence intervals entirely above zero, suggesting consistently positive reports.
- **Antihistamines and cabergoline** lean positive but have wider CIs that cross zero, indicating more variability.
- **Bupropion** (n=18, the largest sample) has a CI centered near zero — reports are split almost evenly.
- **Antipsychotics, dextromethorphan/DXM, and probiotics** have CIs firmly in negative territory.

**What this means:** Treatments on the right (positive) side are worth discussing with a doctor. Wider error bars = less certainty = treat with more caution.

## 7. Treatment Categories: Grouping by Mechanism

Individual drug sample sizes are small. Grouping treatments by mechanism of action increases statistical power and reveals class-level patterns.

In [ ]:
# ── Manual mechanism grouping ──
MECHANISM_MAP = {
    # Dopamine agonists / enhancers
    "cabergoline": "Dopaminergic",
    "pramipexole": "Dopaminergic",
    "bupropion": "Dopaminergic",
    "amphetamine": "Dopaminergic",
    "methylphenidate": "Dopaminergic",
    # PDE5 inhibitors
    "tadalafil": "PDE5 Inhibitor",
    "sildenafil": "PDE5 Inhibitor",
    # Psychedelics
    "microdosing": "Psychedelics",
    "shrooms": "Psychedelics",
    "lsd": "Psychedelics",
    # Antihistamines
    "antihistamine": "Antihistamines",
    "cyproheptadine": "Antihistamines",
    "loratadine": "Antihistamines",
    "cetirizine": "Antihistamines",
    "ketotifen": "Antihistamines",
    "ciproheptadine": "Antihistamines",
    # Diet / lifestyle
    "ketogenic diet": "Diet & Lifestyle",
    "exercise": "Diet & Lifestyle",
    "fasting": "Diet & Lifestyle",
    # Cannabis
    "weed": "Cannabis",
    "cannabis": "Cannabis",
    "marijuana": "Cannabis",
    # Supplements
    "probiotics": "Supplements",
    "magnesium glycinate": "Supplements",
    "magnesium": "Supplements",
    "vitamin c": "Supplements",
    "omega-3 fatty acids": "Supplements",
    "liposomal quercetin": "Supplements",
    "quercetin": "Supplements",
    # Hormonal
    "hcg": "Hormonal",
    "testosterone": "Hormonal",
    "testosterone replacement therapy": "Hormonal",
    # DXM
    "dextromethorphan": "NMDA Antagonist",
    "dxm": "NMDA Antagonist",
    # Other psych drugs
    "gabapentin": "Other Psych Drugs",
    "buspirone": "Other Psych Drugs",
    "antipsychotics": "Other Psych Drugs",
    "olanzapine": "Other Psych Drugs",
    "trazodone": "Other Psych Drugs",
    "benzodiazepines": "Other Psych Drugs",
    "atomoxetine": "Other Psych Drugs",
    "amitriptyline": "Other Psych Drugs",
}

recovery_user["mechanism"] = recovery_user["drug"].str.lower().map(
    {k.lower(): v for k, v in MECHANISM_MAP.items()}
)
recovery_user["mechanism"] = recovery_user["mechanism"].fillna("Other")

mech_summary = recovery_user[recovery_user["mechanism"] != "Other"].groupby("mechanism").agg(
    users=("user_id", "nunique"),
    mean_sentiment=("avg_sentiment", "mean"),
    pct_positive=("outcome", lambda x: (x == "positive").mean() * 100),
    pct_negative=("outcome", lambda x: (x == "negative").mean() * 100),
    pct_mixed=("outcome", lambda x: (x == "mixed/neutral").mean() * 100)
).reset_index().sort_values("mean_sentiment", ascending=False)

mech_display = mech_summary.copy()
mech_display.columns = ["Mechanism", "Users", "Mean Sentiment", "% Positive", "% Negative", "% Mixed"]
mech_display["Mean Sentiment"] = mech_display["Mean Sentiment"].round(2)
mech_display["% Positive"] = mech_display["% Positive"].round(1)
mech_display["% Negative"] = mech_display["% Negative"].round(1)
mech_display["% Mixed"] = mech_display["% Mixed"].round(1)

display(HTML("<h4>Treatment Outcomes by Mechanism of Action</h4>"))
display(mech_display.style.hide(axis='index')
    .background_gradient(subset=["Mean Sentiment"], cmap="RdYlGn", vmin=-1, vmax=1)
    .set_properties(**{"text-align": "center"})
    .set_table_styles([{"selector": "th", "props": [("text-align", "center")]}])
)

In [ ]:
# ── Diverging bar chart: treatment categories ──
mech_plot = mech_summary.sort_values("mean_sentiment", ascending=True).copy()

mechs = mech_plot["mechanism"].values
y_m = np.arange(len(mechs))
pct_pos_m = mech_plot["pct_positive"].values
pct_neg_m = mech_plot["pct_negative"].values
pct_mix_m = mech_plot["pct_mixed"].values
users_m = mech_plot["users"].values

fig, ax = plt.subplots(figsize=(11, 6))
ax.barh(y_m, -pct_mix_m, height=0.6, color="#d5d8dc", label="Mixed/Neutral", edgecolor="white", linewidth=0.5)
ax.barh(y_m, -pct_neg_m, height=0.6, left=-pct_mix_m, color="#e74c3c", label="Negative", edgecolor="white", linewidth=0.5)
ax.barh(y_m, pct_pos_m, height=0.6, color="#2ecc71", label="Positive", edgecolor="white", linewidth=0.5)
ax.axvline(x=0, color="black", linewidth=0.8)

for i in range(len(mechs)):
    ax.text(max(pct_pos_m[i], 3) + 2, i, f"n={users_m[i]}", va="center", fontsize=9, color="#555")

ax.set_yticks(y_m)
ax.set_yticklabels(mechs, fontsize=10)
ax.set_xlabel("% of users", fontsize=11)
ax.set_title("Recovery Outcomes by Treatment Category", fontsize=13, fontweight="bold", pad=12)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{abs(x):.0f}%"))
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=3, frameon=False, fontsize=10)
fig.subplots_adjust(bottom=0.18)
plt.tight_layout()
plt.show()

Grouping by mechanism reveals a clear hierarchy:

1. **Diet & Lifestyle** has the highest positive rate but is based on a small, self-selected sample.
2. **Antihistamines** are broadly positive with reasonable sample size, consistent across multiple specific drugs.
3. **PDE5 Inhibitors** (tadalafil, sildenafil) show net positive sentiment — consistent with their use for sexual dysfunction specifically.
4. **Dopaminergic agents** show a moderate positive lean, though bupropion (the largest contributor) is nearly evenly split.
5. **NMDA Antagonists** (DXM/dextromethorphan) and **Other Psych Drugs** (antipsychotics, gabapentin, etc.) are predominantly negative.

**What this means:** The data supports a pattern: drugs that increase dopamine activity or address sexual dysfunction directly tend to get better reports. Drugs that further modulate serotonin or broadly suppress CNS function tend to get worse reports. Supplements and cannabis fall in between.

## 8. Signal Strength Analysis

Not all reports carry equal weight. The extraction pipeline assigns a signal strength (strong, moderate, weak) based on how clearly the user described their experience. We check whether filtering to strong-signal reports changes the picture.

In [ ]:
# ── Signal strength breakdown for recovery drugs ──
df_rec_signal = df_recovery.copy()
signal_summary = df_rec_signal.groupby("signal_strength").agg(
    reports=("sentiment_score", "count"),
    mean_sentiment=("sentiment_score", "mean"),
    pct_positive=("sentiment_label", lambda x: (x == "positive").mean() * 100)
).reset_index()
signal_summary.columns = ["Signal Strength", "Reports", "Mean Sentiment", "% Positive"]
signal_summary["Mean Sentiment"] = signal_summary["Mean Sentiment"].round(2)
signal_summary["% Positive"] = signal_summary["% Positive"].round(1)

# Strong-signal only: top recovery drugs
df_strong = df_recovery[df_recovery["signal_strength"] == "strong"].copy()
strong_user = df_strong.groupby(["drug", "user_id"]).agg(
    avg_sentiment=("sentiment_score", "mean")
).reset_index()
strong_user["outcome"] = strong_user["avg_sentiment"].apply(
    lambda x: "positive" if x > 0.7 else ("negative" if x < -0.7 else "mixed/neutral")
)

strong_summary = strong_user.groupby("drug").agg(
    users=("user_id", "nunique"),
    mean_sentiment=("avg_sentiment", "mean"),
    pct_positive=("outcome", lambda x: (x == "positive").mean() * 100)
).reset_index().sort_values("users", ascending=False)

strong_display = strong_summary[strong_summary["users"] >= 3].head(15).copy()
strong_display.columns = ["Treatment", "Users (strong signal)", "Mean Sentiment", "% Positive"]
strong_display["Mean Sentiment"] = strong_display["Mean Sentiment"].round(2)
strong_display["% Positive"] = strong_display["% Positive"].round(1)

display(HTML("<h4>Signal Strength Breakdown (Recovery Treatments)</h4>"))
display(signal_summary.style.hide(axis='index')
    .set_properties(**{"text-align": "center"})
    .set_table_styles([{"selector": "th", "props": [("text-align", "center")]}])
)

display(HTML("<h4>Top Recovery Treatments — Strong Signal Only</h4>"))
display(strong_display.style.hide(axis='index')
    .background_gradient(subset=["Mean Sentiment"], cmap="RdYlGn", vmin=-1, vmax=1)
    .set_properties(**{"text-align": "center"})
    .set_table_styles([{"selector": "th", "props": [("text-align", "center")]}])
)

In [ ]:
# ── Signal strength grouped bar ──
fig, ax = plt.subplots(figsize=(7, 4))
sig_colors = {"strong": "#2c3e50", "moderate": "#7f8c8d", "weak": "#bdc3c7"}
bars = ax.bar(
    signal_summary["Signal Strength"],
    signal_summary["Reports"],
    color=[sig_colors.get(s, "#bdc3c7") for s in signal_summary["Signal Strength"]],
    edgecolor="white", linewidth=0.5
)
for bar, pct in zip(bars, signal_summary["% Positive"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            f"{pct:.0f}% pos", ha="center", fontsize=10, color="#555")
ax.set_ylabel("Number of reports", fontsize=11)
ax.set_title("Report Volume by Signal Strength (Recovery Treatments)",
             fontsize=12, fontweight="bold", pad=10)
plt.tight_layout()
plt.show()

The signal strength breakdown shows that most recovery-treatment reports are strong- or moderate-signal, giving reasonable confidence in the sentiment labels. The positive rate is roughly similar across signal strengths for recovery treatments, suggesting weak-signal reports do not introduce major distortion.

## 9. Recommendations

Treatments are grouped into three tiers based on the strength of the available evidence. Tier placement considers both sample size and the direction/consistency of outcomes.

- **Strong evidence**: n >= 8 users and a clear directional signal
- **Moderate evidence**: n >= 5 users or consistent class-level pattern
- **Preliminary**: n < 5 users or emerging signals worth monitoring

All SSRIs/SNRIs are excluded — they caused PSSD and their negative sentiment reflects the condition's origin, not recovery outcomes.

In [ ]:
# ── Tiered recommendations ──
recommendations = [
    # Strong evidence
    {"Tier": "Strong", "Treatment": "Bupropion", "Users": 18, "% Positive": 38.9,
     "Note": "Largest sample. Dopaminergic antidepressant — mixed results but often tried first. Does not worsen sexual function like SSRIs."},
    {"Tier": "Strong", "Treatment": "Cabergoline", "Users": 9, "% Positive": 44.4,
     "Note": "Dopamine D2 agonist. Net positive lean with n=9. Used off-label for hyperprolactinemia-related sexual dysfunction."},
    {"Tier": "Strong", "Treatment": "Buspirone", "Users": 8, "% Positive": 37.5,
     "Note": "5-HT1A partial agonist. Evenly split outcomes. Sometimes added to SSRIs to counter sexual side effects."},
    # Moderate evidence
    {"Tier": "Moderate", "Treatment": "Antihistamines", "Users": 6, "% Positive": 66.7,
     "Note": "Class-level positive signal across multiple drugs (loratadine, cetirizine, ketotifen). Serotonin-antagonist properties may be relevant."},
    {"Tier": "Moderate", "Treatment": "Tadalafil", "Users": 5, "% Positive": 80.0,
     "Note": "PDE5 inhibitor. Strong positive signal for erectile/arousal symptoms specifically. Does not address emotional blunting."},
    {"Tier": "Moderate", "Treatment": "Microdosing (psilocybin)", "Users": 5, "% Positive": 80.0,
     "Note": "Sub-perceptual psychedelic dosing. Strong positive reports but small sample. 5-HT2A agonism may counteract SSRI effects."},
    {"Tier": "Moderate", "Treatment": "Ketogenic diet", "Users": 5, "% Positive": 80.0,
     "Note": "Dietary intervention with strong positive signal. Mechanism unclear but neuroinflammation reduction is hypothesized."},
    {"Tier": "Moderate", "Treatment": "Pramipexole", "Users": 5, "% Positive": 40.0,
     "Note": "Dopamine D2/D3 agonist. More mixed than cabergoline but same mechanism class."},
    # Preliminary
    {"Tier": "Preliminary", "Treatment": "Ketotifen", "Users": 3, "% Positive": 66.7,
     "Note": "Antihistamine/mast cell stabilizer. Small sample but consistent with antihistamine class signal."},
    {"Tier": "Preliminary", "Treatment": "Quercetin", "Users": 3, "% Positive": 66.7,
     "Note": "Flavonoid with anti-inflammatory and antihistamine properties. Consistent with antihistamine class pattern."},
    {"Tier": "Preliminary", "Treatment": "Exercise", "Users": 2, "% Positive": 100.0,
     "Note": "Universal health recommendation. Both users reported benefit. No downside risk."},
    {"Tier": "Preliminary", "Treatment": "Low dose naltrexone", "Users": 3, "% Positive": 0.0,
     "Note": "All 3 users reported mixed outcomes. Monitored as it works well in other conditions (Long COVID)."},
]

rec_df = pd.DataFrame(recommendations)

tier_colors = {"Strong": "#27ae60", "Moderate": "#f39c12", "Preliminary": "#95a5a6"}

def color_tier(val):
    color = tier_colors.get(val, "")
    return f"background-color: {color}; color: white; font-weight: bold;" if color else ""

display(HTML("<h4>Tiered Treatment Recommendations for PSSD Recovery</h4>"))
display(HTML("<p><em>SSRIs/SNRIs excluded (causal-context contamination). See Section 3 for causal drugs.</em></p>"))
display(rec_df.style.hide(axis='index')
    .map(color_tier, subset=["Tier"])
    .set_properties(**{"text-align": "left"})
    .set_properties(subset=["Tier", "Users", "% Positive"], **{"text-align": "center"})
    .set_table_styles([
        {"selector": "th", "props": [("text-align", "center")]},
        {"selector": "td", "props": [("max-width", "500px")]}
    ])
)

In [ ]:
# ── Visual summary: Forest plot of recommended treatments color-coded by tier ──
rec_forest = []
rec_drugs_map = {
    "Bupropion": "bupropion", "Cabergoline": "cabergoline", "Buspirone": "buspirone",
    "Antihistamines": "antihistamine", "Tadalafil": "tadalafil",
    "Microdosing (psilocybin)": "microdosing", "Ketogenic diet": "ketogenic diet",
    "Pramipexole": "pramipexole", "Ketotifen": "ketotifen",
    "Quercetin": "quercetin", "Low dose naltrexone": "low dose naltrexone"
}

for rec in recommendations:
    treatment_name = rec["Treatment"]
    drug_key = rec_drugs_map.get(treatment_name)
    if drug_key is None:
        continue
    scores = recovery_user[recovery_user["drug"].str.lower() == drug_key.lower()]["avg_sentiment"].values
    if len(scores) < 2:
        continue
    mean = scores.mean()
    se = sp_stats.sem(scores)
    ci = se * sp_stats.t.ppf(0.975, len(scores) - 1)
    rec_forest.append({
        "treatment": treatment_name, "mean": mean, "ci": ci,
        "n": len(scores), "tier": rec["Tier"]
    })

rec_forest_df = pd.DataFrame(rec_forest).sort_values("mean", ascending=True)

fig, ax = plt.subplots(figsize=(10, max(5, len(rec_forest_df) * 0.5)))
y_rec = np.arange(len(rec_forest_df))

for i, row in enumerate(rec_forest_df.itertuples()):
    color = tier_colors.get(row.tier, "#95a5a6")
    ax.errorbar(row.mean, i, xerr=row.ci, fmt="none", ecolor="#777",
                elinewidth=1.5, capsize=4, zorder=1)
    ax.scatter(row.mean, i, c=color, s=100, zorder=2, edgecolors="white", linewidth=1)
    ax.text(1.2, i, f"n={row.n}", va="center", fontsize=9, color="#777")

ax.axvline(x=0, color="gray", linestyle="--", alpha=0.5, linewidth=1)
ax.set_yticks(y_rec)
ax.set_yticklabels(rec_forest_df["treatment"].values, fontsize=10)
ax.set_xlabel("Mean Sentiment Score (95% CI)", fontsize=11)
ax.set_title("Recommended Treatments — Tiered by Evidence Strength",
             fontsize=13, fontweight="bold", pad=12)
ax.set_xlim(-1.4, 1.6)

# Legend for tiers
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor="#27ae60", markersize=10, label="Strong evidence"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="#f39c12", markersize=10, label="Moderate evidence"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="#95a5a6", markersize=10, label="Preliminary"),
]
ax.legend(handles=legend_elements, loc="upper center", bbox_to_anchor=(0.5, -0.12),
          ncol=3, frameon=False, fontsize=10)
fig.subplots_adjust(bottom=0.15)
plt.tight_layout()
plt.show()

The tiered forest plot shows recommended treatments color-coded by evidence strength. Key takeaways:

- **Tadalafil, microdosing, and ketogenic diet** have the highest mean sentiment with CIs above zero, but all are n=5 (moderate evidence).
- **Cabergoline** (n=9) is the best-supported dopaminergic option with a positive lean.
- **Bupropion** (n=18) has the tightest CI but sits near zero — it works for some but not others.
- **Buspirone** (n=8) is similar to bupropion: worth trying, but expect mixed results.

**What this means for patients:** There is no silver bullet for PSSD in this data. The strongest positive signals come from PDE5 inhibitors (for sexual symptoms specifically), psychedelic microdosing, dietary changes, and dopamine agonists. Discuss these options with your doctor, starting with the lower-risk interventions (diet, exercise, antihistamines) before moving to prescription options.

## Summary

### Key Findings

**Question:** What treatments help or harm people with PSSD?

**The data:** 902 treatment reports from 220 users on r/PSSD, covering March–April 2026. Overall sentiment is 62% negative, driven by SSRI/SNRI mentions (the drugs that caused the condition).

**What caused PSSD (excluded from recovery analysis):**
- SSRIs/SNRIs account for the majority of treatment mentions and are overwhelmingly negative (85–100% negative). Sertraline, fluoxetine, paroxetine, and escitalopram are the most frequently cited causative agents.

**What shows promise for recovery:**
- **PDE5 inhibitors (tadalafil):** 71–80% positive (n=5). Directly addresses sexual dysfunction symptoms. Best for erectile/arousal complaints specifically.
- **Psychedelic microdosing:** 78% positive (n=5). 5-HT2A agonism may help reverse SSRI-induced changes. Legal access varies.
- **Ketogenic diet:** 83% positive (n=5). Low-risk dietary intervention. Mechanism unclear but neuroinflammation reduction hypothesized.
- **Antihistamines (class-wide):** 67% positive (n=6). Multiple drugs in this class show consistent benefit. Serotonin-antagonist properties may be relevant.
- **Dopamine agonists (cabergoline, pramipexole):** 40–44% positive (n=5–9). Moderate benefit. Cabergoline has the strongest signal in this class.
- **Bupropion:** 39% positive (n=18). The most-tried non-SSRI treatment. Mixed results but does not worsen sexual function.

**What appears harmful:**
- **Antipsychotics:** 88% negative. Likely worsen sexual and emotional symptoms.
- **DXM/Dextromethorphan:** 78–80% negative. NMDA antagonism does not appear helpful.
- **Probiotics:** 70% negative. No evidence of benefit for PSSD specifically.
- **Gabapentin:** 60% negative. CNS depressant that may compound symptoms.

### Caveats

- **Small sample sizes:** Maximum 18 users per non-SSRI drug. Statistical power is limited and most tests do not reach significance.
- **1-month snapshot:** Data covers ~1 month of r/PSSD activity. Longer observation windows would capture more treatments and users.
- **Self-selection bias:** People who post about treatments may be more motivated, more distressed, or have more extreme experiences than typical PSSD patients.
- **No clinical validation:** These are patient-reported community outcomes, not controlled trial results.
- **Canonicalization gaps:** Some treatments may be split across multiple names (e.g., DXM/dextromethorphan, weed/cannabis/marijuana) despite attempted deduplication.

### Next Steps

1. **Expand the dataset:** Scrape 3–6 months of r/PSSD data to increase per-drug sample sizes into statistically meaningful ranges (n >= 30).
2. **Symptom-specific analysis:** PSSD has multiple symptom domains (sexual dysfunction, emotional blunting, cognitive fog). Different treatments may help different symptoms.
3. **Longitudinal tracking:** Some treatments may show improvement over time (weeks/months). A time-series analysis could capture delayed benefits.
4. **Cross-community comparison:** Compare PSSD treatment patterns with r/anhedonia and r/DPDR, which share symptom overlap.

---

*Based on self-selected Reddit posts. Users who never posted about a treatment are not represented. Results reflect reporting patterns, not population-level treatment effects.*

In [ ]:
conn.close()